In [1]:
%load_ext autoreload
%autoreload 2


# Import Libraries

In [2]:
#!pip install "numpy<1.24" --force-reinstall

In [3]:
#!pip install --force-reinstall catboost

In [2]:
import numpy as np
np.__version__

'1.26.4'

In [2]:
import os

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import spacy
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
import catboost as cat
from catboost import Pool, CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import make_pipeline

import gensim.models
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


# Read Data

In [6]:
#from google.colab import drive
#drive.mount('/content/drive')


In [7]:
#train_path = '/content/drive/My Drive/train.csv'
#test_path = '/content/drive/My Drive/test.csv'

In [20]:
nltk.download("stopwords")
nltk_stopwords = nltk.corpus.stopwords.words("russian")
print(f'NLTK ru stopwords size: {len(nltk_stopwords)}', end='\n\n')
' '.join(nltk_stopwords)

NLTK ru stopwords size: 151



[nltk_data] Downloading package stopwords to
[nltk_data]     /home/vaa2804/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'

In [21]:
train_path = '../../train.csv'
test_path = '../../test.csv'

In [4]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(f"Number of rows and columns in the train data set: {train.shape}")
print(f"Number of rows and columns in the test data set: {test.shape}")
train.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [5]:
train['rate'].value_counts()

rate
5    26069
4     9922
3     6126
1     4138
2     2410
Name: count, dtype: int64

In [22]:
le = LabelEncoder()

train.rate = le.fit_transform(train.rate)
train.head()

,rate,text,cleaned_text
0,3,Очень понравилось. Были в начале марта с соба...,очень понравиться были в начало март с собака ...
1,4,В целом магазин устраивает.\nАссортимент позво...,в целое магазин устраивать ассортимент позволя...
2,4,"Очень хорошо что открылась 5 ка, теперь не над...",очень хороший что открыться ка теперь не надо ...
3,2,Пятёрочка громко объявила о том как она заботи...,громко объявить о том как она заботиться о пен...
4,2,"Тесно, вечная сутолока, между рядами трудно ра...",тесно вечный сутолока между ряд трудный разойт...


In [26]:
nlp = spacy.load("ru_core_news_sm")
#stopwords = nlp.Defaults.stop_words
nlp.stopwords = nltk_stopwords[:100]

In [27]:
print(f'NLP ru stopwords size: {len(nlp.stopwords)}', end='\n\n')
' '.join(nlp.stopwords)

NLP ru stopwords size: 100



'и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один'

In [28]:
train['cleaned_text'] = train['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [29]:
train_vec = TfidfVectorizer(ngram_range = (1,1), max_features = 500)
train_bow = train_vec.fit_transform(train['cleaned_text'])

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_bow, train['rate'], shuffle = True, random_state=2025)

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes = classes, y = y_train)
class_weights=dict(zip(classes,weights))
class_weights

In [ ]:
model = CatBoostClassifier(loss_function='MultiClass',  learning_rate = 0.05, iterations = 1000, class_weights = class_weights,  early_stopping_rounds= 200,  random_seed=42)


In [ ]:
model.fit(X_train,y_train, eval_set=(X_test, y_test),verbose = False)

In [ ]:
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))

In [ ]:
test['cleaned_text'] = test['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and not token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [ ]:
test_vec = TfidfVectorizer(ngram_range = (1,1), max_features = 500)
test_bow = test_vec.fit_transform(test['cleaned_text'])
predictions = model.predict(test_bow)

In [ ]:
sample_submission_path = '../../sample_submission.csv'
submission = pd.read_csv(sample_submission_path)
submission["rate"] = le.inverse_transform(predictions)
submission.head()

In [ ]:
submission_path = '../../submission.csv'
submission.to_csv(submission_path, index=False)

In [16]:
x_array = train['cleaned_text'].to_numpy()
x_array=x_array.tolist()

In [24]:
model = gensim.models.Word2Vec(
    sentences=x_array,
    vector_size=256, # default = 100
    window=3, # default = 5
    min_count=3,
    sg=1, # Training algorithm: 1 for skip-gram; otherwise CBOW
    hs=0, #  If 1, hierarchical softmax will be used for model training. If 0, and negative is non-zero, negative sampling will be used.
    negative=2, # If > 0, negative sampling will be used, if set to 0, no negative sampling is used.
    epochs=25, # Number of iterations (epochs) over the corpus
    seed=2023,
)

In [18]:
def text_to_vector(text):
    vectors = [model.wv[word] for word in text if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [25]:
X = np.array([text_to_vector(text) for text in x_array])
X_train, X_test, y_train, y_test = train_test_split(X, train['rate'], shuffle = True, random_state=2025)

In [20]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes = classes, y = y_train)
class_weights=dict(zip(classes,weights))
class_weights

{1: 2.324713375796178,
 2: 3.9975903614457833,
 3: 1.596937212863706,
 4: 0.9847025495750709,
 5: 0.3734192756292204}

In [27]:
model = CatBoostClassifier(loss_function='MultiClass',  learning_rate = 0.05, iterations = 1000, class_weights = class_weights,  early_stopping_rounds= 200,  random_seed=42)


In [28]:
model.fit(X_train,y_train, eval_set=(X_test, y_test),verbose = False)

In [29]:
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))


              precision    recall  f1-score   support

           1       0.26      0.52      0.34       998
           2       0.09      0.16      0.11       584
           3       0.23      0.25      0.24      1555
           4       0.30      0.27      0.28      2509
           5       0.76      0.60      0.67      6521

    accuracy                           0.46     12167
   macro avg       0.33      0.36      0.33     12167
weighted avg       0.52      0.46      0.48     12167



In [ ]:
params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [300, 500, 700],
    'l2_leaf_reg': [1, 3, 5]
}

In [ ]:
grid_search = GridSearchCV(model, params, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)

# Preparing the data and creating Catboost model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train, train['rate'], shuffle = True, random_state=2025)

In [ ]:
X_train.head()

,rate,text,cleaned_text
38373,5,"Хороший выбор товаров, побольше, чем у соседне...",хороший выбор товар большой соседний малый про...
37650,5,"Хорошая пятёрочка, всегда всё на своих местах....",хороший место небольшой парковка
11122,3,Последнее время очень упал общий уровень магаз...,последний время упасть общий уровень магазин с...
1978,5,"Хороший магазин! Отличное местоположение, ассо...",хороший магазин отличный местоположение ассорт...
42401,5,Обожаю кофе в этом магазине,обожать кофе магазин


In [ ]:
X_train = X_train.drop(['rate','text'],axis = 1)
X_test = X_test.drop(['rate','text'],axis = 1)

In [ ]:
model = CatBoostClassifier( iterations = 1000, class_weights = class_weights,  early_stopping_rounds= 200, random_seed=42)


In [ ]:
model.fit(X_train, y_train, text_features=[0],verbose=False,eval_set=(X_test, y_test) )

In [ ]:
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           1       0.48      0.62      0.54       998
           2       0.17      0.24      0.20       584
           3       0.33      0.38      0.35      1555
           4       0.40      0.41      0.40      2509
           5       0.84      0.73      0.78      6521

    accuracy                           0.58     12167
   macro avg       0.44      0.48      0.45     12167
weighted avg       0.62      0.58      0.60     12167



Оптимизация гиперпараметров

In [ ]:
params = {
    'text_features' : [[0]],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [300, 500, 700],
    'l2_leaf_reg': [1, 3, 5]
}

In [ ]:
grid_search = GridSearchCV(model, params, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)

0:	learn: 1.6034776	total: 2.05s	remaining: 10m 12s
1:	learn: 1.5976230	total: 3.6s	remaining: 8m 56s
2:	learn: 1.5919506	total: 5.39s	remaining: 8m 53s
3:	learn: 1.5863597	total: 7.18s	remaining: 8m 51s
4:	learn: 1.5809381	total: 8.59s	remaining: 8m 26s
5:	learn: 1.5759222	total: 9.96s	remaining: 8m 7s
6:	learn: 1.5706248	total: 11.3s	remaining: 7m 54s
7:	learn: 1.5657839	total: 12.9s	remaining: 7m 52s
8:	learn: 1.5614700	total: 14.3s	remaining: 7m 41s
9:	learn: 1.5570528	total: 15.7s	remaining: 7m 35s
10:	learn: 1.5526024	total: 17s	remaining: 7m 25s
11:	learn: 1.5487421	total: 18s	remaining: 7m 13s
12:	learn: 1.5450480	total: 19.4s	remaining: 7m 8s
13:	learn: 1.5411329	total: 20.9s	remaining: 7m 6s
14:	learn: 1.5368641	total: 22.3s	remaining: 7m 3s
15:	learn: 1.5325636	total: 23.5s	remaining: 6m 57s
16:	learn: 1.5284162	total: 24.9s	remaining: 6m 54s
17:	learn: 1.5243719	total: 26.5s	remaining: 6m 55s
18:	learn: 1.5204316	total: 28s	remaining: 6m 53s
19:	learn: 1.5165897	total: 29.3

In [ ]:
X_train = train["cleaned_text"]
y_train = train["rate"]

X_test = test["text"]


model = CatBoostClassifier(
    iterations=100,
    depth=5,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    text_features=[0],
    verbose=True
)

Learning rate set to 0.5
0:	learn: 1.1115052	total: 1.53s	remaining: 2m 31s
1:	learn: 1.0314105	total: 3.12s	remaining: 2m 33s
2:	learn: 0.9927610	total: 4.69s	remaining: 2m 31s
3:	learn: 0.9725062	total: 6.7s	remaining: 2m 40s
4:	learn: 0.9606198	total: 8.01s	remaining: 2m 32s
5:	learn: 0.9549722	total: 9.28s	remaining: 2m 25s
6:	learn: 0.9518823	total: 10.4s	remaining: 2m 18s
7:	learn: 0.9487206	total: 11.6s	remaining: 2m 13s
8:	learn: 0.9447613	total: 12.9s	remaining: 2m 10s
9:	learn: 0.9433536	total: 13.9s	remaining: 2m 5s
10:	learn: 0.9420060	total: 15.1s	remaining: 2m 2s
11:	learn: 0.9391503	total: 16.4s	remaining: 2m
12:	learn: 0.9360403	total: 17.9s	remaining: 1m 59s
13:	learn: 0.9351435	total: 18.9s	remaining: 1m 55s
14:	learn: 0.9339834	total: 20.1s	remaining: 1m 53s
15:	learn: 0.9328480	total: 21.1s	remaining: 1m 50s
16:	learn: 0.9301511	total: 22.4s	remaining: 1m 49s
17:	learn: 0.9295311	total: 23.5s	remaining: 1m 47s
18:	learn: 0.9286594	total: 24.7s	remaining: 1m 45s
19:	

# Predict

In [64]:
test['cleaned_text'] = test['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and not token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [66]:
test_vec = TfidfVectorizer(ngram_range = (1,1), max_features = 500)
test_bow = vec.fit_transform(test['cleaned_text'])
predictions = model.predict(test_bow)

In [ ]:
# Preparing data in Pool format
dataset_test = Pool(
    data=X_test,
    text_features=[0]
)
predict_classes = model.predict(dataset_test)
predictions = predict_classes

# Create submission

In [67]:
sample_submission_path = '/content/drive/My Drive/sample_submission.csv'
submission = pd.read_csv(sample_submission_path)
submission["rate"] = predictions
submission.head()

,index,rate
0,0,1
1,1,1
2,2,3
3,3,5
4,4,3


In [70]:
submission_path = '/content/drive/My Drive/submission.csv'
submission.to_csv(submission_path, index=False)